In [1]:
import numpy as np

In [43]:
#Linear Regression: n-dimensional polynomial fitting limited to terms of degree 1
class linreg():
    def __init__(self, weights):
        self.weights = np.asarray(weights)
        self.learning_rate = 0.001

    def predict(self, features):
        assert isinstance(features, np.ndarray), "Features is not a numpy array"
        assert len(features) == len(self.weights)-1, f"Length of features ({len(features)} does not match length of weights-1, {len(self.weights)-1})"
        f = np.concatenate(([1], features))
        return sum(self.weights*f)

    #returns an array of loss per feature
    def stochastic_loss(self, features, target): #compute loss for a single example
        p = self.predict(features)
        f = np.concatenate(([1], features))
        return self.learning_rate*(p-target)*f

    #returns an array of cumulative loss per feature
    def batch_loss(self, examples):
        adjustment = 0.
        for example in examples:
            features, target = example
            adjustment += self.stochastic_loss(features, target) #alpha inside loop
        return adjustment

    def step_bgd(self, examples):
        adjustment = self.batch_loss(examples)
        self.weights = np.subtract(self.weights, adjustment)
        return

    def step_sgd(self, examples):
        for example in examples:
            features, target = example
            adjustment = self.stochastic_loss(features, target)
            self.weights = np.subtract(self.weights, adjustment)
        return
        

In [44]:
#generate examples
#Set noise_std to see if the model recovers weights-->coeffs exactly
def generate_dataset(n_features, n_samples, coeffs, noise_std=0.5, x_range=(-10, 10), rng=None):
    rng = rng or np.rnadom.default_rng()
    coeffs = np.asarray(coeffs)
    assert len(coeffs) == n_features+1

    X = rng.uniform(x_range[0], x_range[1], size=(n_samples, n_features))
    bias, w = coeffs[0], coeffs[1:]
    y_clean = X @ w +  bias #'@' is numpy matrix multiplication
    noise = rng.normal(0, noise_std, size=n_samples)
    y=y_clean+noise

    return [(X[i], y[i]) for i in range(n_samples)]

In [62]:
#run test
rng = np.random.default_rng(seed=6)
coefficients = np.array([0, 1.5, 10, 2, -3, 0.5, 1])

trainset = generate_dataset(n_features=6, n_samples=100, coeffs=coefficients, rng=rng)
testset = generate_dataset(n_features=6, n_samples=20,  coeffs=coefficients, rng=rng)

model = linreg(np.zeros(7))

for epoch in range(50):
    model.step_bgd(trainset)

#evaluate
mse = np.mean([(model.predict(f)-t)**2 for f, t, in testset])
print("Test MSE:", mse, "Learned Weights:", model.weights)

Test MSE: 2.394505907125095e+58 Learned Weights: [ 6.93996526e+24  1.31952448e+28 -1.00696214e+28 -1.50984942e+28
  8.12305241e+27  1.08692174e+28  1.17216674e+28]


In [61]:
model.step_bgd(trainset)
print("Learned Weights:", model.weights)

Learned Weights: [ 5.65504748e+33  1.07184287e+37 -8.17937408e+36 -1.22636361e+37
  6.59603736e+36  8.82885986e+36  9.52215990e+36]
